# 21 Aggregate stock features to funds

目标：只汇合持仓分支和股票分支；每个特征按自身有效持仓市值集合重新归一化。

- 输入：13 号月度持仓、20 号股票排名/PCA
- 正式输出：fund_stock_features.parquet、逐特征覆盖率
- 主键：wficn+month
- 上游依赖：13、20
- smoke 输出与正式输出隔离；报告会明确标注运行模式。

In [ ]:
from __future__ import annotations
import json, os, sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config_utils import load_project_config
from src.path_utils import find_project_root
from src.runtime_paths import (
    clean_output_path,
    execution_context,
    install_clean_read_guard,
)

ROOT = find_project_root(ROOT)
EXECUTION_CONTEXT = execution_context()
install_clean_read_guard(ROOT)
CONFIG = load_project_config(ROOT)
PARAMS = CONFIG["parameters"]
PATHS = CONFIG["resolved_paths"]
RUN_MODE = os.getenv("KANIEL_RUN_MODE", "smoke").strip().lower()
SMOKE = RUN_MODE != "full"
if RUN_MODE == "full":
    FULL_RUN_TMP = (
        clean_output_path(ROOT, "data/interim/_full_run_tmp")
        if EXECUTION_CONTEXT == "clean_rebuild_compute"
        else PATHS["full_run_tmp"]
    )
    PATHS["full_run_tmp"] = FULL_RUN_TMP
    FULL_RUN_TMP.mkdir(parents=True, exist_ok=True)
    os.environ["TMP"] = str(FULL_RUN_TMP)
    os.environ["TEMP"] = str(FULL_RUN_TMP)
    os.environ["TMPDIR"] = str(FULL_RUN_TMP)
    os.environ["POLARS_TEMP_DIR"] = str(FULL_RUN_TMP)
STARTED_AT = time.time()
print({
    "project_root": str(ROOT),
    "run_mode": RUN_MODE,
    "execution_context": EXECUTION_CONTEXT,
})


In [ ]:
import os
import duckdb
from src.atomic_write import duckdb_copy_csv_atomic, duckdb_copy_parquet_atomic
from src.runtime_paths import execution_context, output_path, upstream_path

names=['r2_1', 'r12_2', 'r12_7', 'r36_13', 'ST_Rev', 'LT_Rev', 'Investment', 'NOA', 'DPI2A', 'NI', 'PROF', 'ATO', 'CTO', 'FC2Y', 'OP', 'PM', 'RNA', 'ROA', 'ROE', 'SGA2S', 'D2A', 'AC', 'OA', 'OL', 'PCM', 'A2ME', 'BEME', 'C', 'CF', 'CF2P', 'D2P', 'E2P', 'Q', 'S2P', 'Lev', 'AT', 'Beta', 'IdioVol', 'LME', 'LTurnover', 'MktBeta', 'Rel2High', 'Resid_Var', 'Spread', 'SUV', 'Variance']
def esc(path): return str(path).replace("'","''")
h_path=upstream_path(ROOT,RUN_MODE,"13_expand_report_holdings_to_monthly","data/processed/holdings/monthly_holdings_weights.parquet")
s_path=upstream_path(ROOT,RUN_MODE,"20_rank_and_pca_impute_stock_features","data/processed/stock/stock_46_ranked_imputed.parquet")
con=duckdb.connect()
con.execute(f"SET memory_limit='{PARAMS['duckdb_memory_limit']}'")
con.execute(f"SET temp_directory='{esc(PATHS['full_run_tmp'])}'")
con.execute("SET threads=1")
con.execute("SET preserve_insertion_order=true")
con.execute(f"""CREATE TEMP VIEW merged AS
 SELECT h.wficn,h."month",h.permno,h.holding_weight,
        s.permno IS NOT NULL AS _stock_match,
        s.* EXCLUDE(permno,"month")
 FROM read_parquet('{esc(h_path)}') h
 LEFT JOIN read_parquet('{esc(s_path)}') s USING(permno,"month")""")
# A single 92-aggregate hash table exceeds the locked DuckDB memory budget on
# the 202M-row holdings panel.  Compute one canonical total and bounded batches
# of feature-specific numerator/denominator pairs.  Every reduction still uses
# the identical valid set, stable permno order, float64 fsum and one thread.
batch_size=8
total_stage=PATHS["full_run_tmp"]/"21_canonical_total_market_value.parquet"
batch_paths=[]
feature_stage=PATHS["full_run_tmp"]/"21_fund_46_stock_features_staged.parquet"
total_stage.unlink(missing_ok=True); feature_stage.unlink(missing_ok=True)
duckdb_copy_parquet_atomic(
    con,
    f'SELECT wficn,"month", '
    f'fsum(CASE WHEN holding_weight>0 THEN CAST(holding_weight AS DOUBLE) END) '
    f'AS total_holding_weight FROM read_parquet(\'{esc(h_path)}\') '
    f'GROUP BY wficn,"month" ORDER BY wficn,"month"',
    total_stage,
)
for start in range(0,len(names),batch_size):
    batch=names[start:start+batch_size]
    expr=[]
    for feature in batch:
        valid=f'holding_weight>0 AND "{feature}" IS NOT NULL AND isfinite("{feature}")'
        denom=f'fsum(CASE WHEN {valid} THEN CAST(holding_weight AS DOUBLE) END)'
        numerator=f'fsum(CASE WHEN {valid} THEN CAST(holding_weight AS DOUBLE)*CAST("{feature}" AS DOUBLE) END)'
        expr.extend([
            f'{numerator}/nullif({denom},0) "{feature}"',
            f'{denom} "{feature}_valid_holding_weight"',
        ])
    batch_path=PATHS["full_run_tmp"]/f"21_feature_batch_{start//batch_size:02d}.parquet"
    batch_path.unlink(missing_ok=True)
    duckdb_copy_parquet_atomic(
        con,
        'SELECT wficn,"month",'+",".join(expr)+' FROM merged GROUP BY wficn,"month" ORDER BY wficn,"month"',
        batch_path,
    )
    batch_paths.append(batch_path)
relations=[f"read_parquet('{esc(total_stage)}') t"]
joins=[]
for i,path in enumerate(batch_paths):
    alias=f"b{i}"
    relations.append(f"read_parquet('{esc(path)}') {alias}")
    joins.append(f"JOIN {relations[-1]} USING(wficn,\"month\")")
value_expr=[f'b{i//batch_size}."{feature}"' for i,feature in enumerate(names)]
coverage_expr=[
    f'b{i//batch_size}."{feature}_valid_holding_weight"/nullif(t.total_holding_weight,0) "{feature}_coverage"'
    for i,feature in enumerate(names)
]
combined_query=(
    'SELECT wficn,"month",'+",".join(value_expr+coverage_expr)+
    ' FROM '+relations[0]+' '+ " ".join(joins)
)
con.execute(f"CREATE TEMP VIEW combined AS {combined_query}")
out=output_path(ROOT,RUN_MODE,"21_aggregate_stock_features_to_funds","data/processed/fund/fund_46_stock_features.parquet")
aud=output_path(ROOT,RUN_MODE,"21_aggregate_stock_features_to_funds","data/processed/fund/fund_46_stock_feature_coverage.parquet")
join_audit=output_path(ROOT,RUN_MODE,"21_aggregate_stock_features_to_funds","data/audit/21_join_audit.csv")
duckdb_copy_parquet_atomic(con,'SELECT wficn,"month",'+",".join(f'"{c}"' for c in names)+' FROM combined ORDER BY wficn,"month"',feature_stage)
duckdb_copy_parquet_atomic(con,'SELECT wficn,"month",'+",".join(f'"{c}_coverage"' for c in names)+' FROM combined ORDER BY wficn,"month"',aud)
holding_count=int(con.execute(f"SELECT count(*) FROM read_parquet('{esc(h_path)}')").fetchone()[0])
stock_count=int(con.execute(f"SELECT count(*) FROM read_parquet('{esc(s_path)}')").fetchone()[0])
merged_count=int(con.execute("SELECT count(*) FROM merged").fetchone()[0])
fund_count=int(con.execute("SELECT count(*) FROM combined").fetchone()[0])
duplicate_count=int(con.execute("""SELECT coalesce(sum(n-1),0) FROM (
 SELECT count(*) n FROM combined GROUP BY wficn,"month" HAVING count(*)>1)""").fetchone()[0])
unmatched_count=int(con.execute("SELECT count(*) FROM merged WHERE NOT _stock_match").fetchone()[0])
duckdb_copy_csv_atomic(con,f"""SELECT {holding_count} left_rows,{stock_count} right_rows,
 {merged_count} result_rows,{merged_count/holding_count if holding_count else 1.0} expansion_ratio,
 {unmatched_count} unmatched_holding_rows""",join_audit)
assert merged_count==holding_count and duplicate_count==0 and len(names)==46
con.close()
out.parent.mkdir(parents=True,exist_ok=True); os.replace(feature_stage,out)
total_stage.unlink(missing_ok=True)
for path in batch_paths: path.unlink(missing_ok=True)
INPUTS_USED=["monthly_holdings_weights.parquet","stock_46_ranked_imputed.parquet"]; OUTPUTS_WRITTEN=[str(p.relative_to(ROOT)) for p in [out,aud]]
OUTPUTS_WRITTEN.append(str(join_audit.relative_to(ROOT)))
INPUT_COUNTS={"holdings":holding_count,"stock_features":stock_count}; OUTPUT_COUNTS={"fund_months":fund_count}; PRIMARY_KEY=["wficn","month"]; DUPLICATE_KEY_COUNT=duplicate_count
EXCLUDED_ROW_COUNT=unmatched_count; EXCEPTION_FILE=str(join_audit.relative_to(ROOT)); STAGE_STATUS="passed"
STAGE_NOTES=["This stage merges only holdings and stock branches.","Each feature uses its own identical valid holding set for its numerator and denominator; every feature is separately re-normalized and no coverage threshold was invented.","Stable permno order, float64 compensated reductions and one critical thread make the fund-feature aggregation deterministic; no rounding is applied."]


In [ ]:
from src.audit_utils import write_run_summary
summary_path = write_run_summary(
    ROOT, "21_aggregate_stock_features_to_funds", stage="21", started_at=STARTED_AT,
    input_files=INPUTS_USED, output_files=OUTPUTS_WRITTEN,
    input_row_counts=INPUT_COUNTS, output_row_counts=OUTPUT_COUNTS,
    primary_key=PRIMARY_KEY, duplicate_key_count=DUPLICATE_KEY_COUNT,
    excluded_row_count=EXCLUDED_ROW_COUNT, exception_file=EXCEPTION_FILE,
    status=STAGE_STATUS, notes=STAGE_NOTES,
)
print(summary_path)
